# Negotiate, Then Solve: training the local model

LoRA fine-tuning of **Qwen3.5-2B** (4B needs a bf16 GPU), then export to GGUF for
local inference. Two datasets work unchanged:

* **compiler only** (`data/compiler/`, from `language.compiler`): requests to the same
  `ParseOutput` JSON as the Gemini System Two parser;
* **multi-task** (`data/multitask/`, from `training.distill export`): the compiler data
  plus reading negotiation replies, grounded explanations and the policy agent,
  each with the system prompt the deployed system sends. Rows carry a `task`
  field, and the check below reports each task separately.

**Before running**

1. On your machine: `uv run python -m language.compiler` writes `data/compiler/`;
   for the multi-task model run the `training.distill` stages (see its docstring) and
   `uv run python -m training.distill export`, which writes `data/multitask/`. Upload
   the folder as a Kaggle dataset and add it to this notebook (*Add Input*).
   With both attached, the multi-task one is used.
2. *Settings*: **Accelerator: GPU T4** (x1 or x2; only one is used), **Internet: on**.
3. Run with `TRIAL = True` first (20 steps, a few minutes) to check that
   installation, float32 on the T4 and memory are fine. Then set `TRIAL = False`
   and *Run all*.

**Precision on a T4:** Unsloth advises against 4-bit (QLoRA) training for
Qwen3.5, and Qwen3.5 does not train in fp16, so on a T4 (no bf16) Unsloth uses
float32. The 2B model then needs about 8 GB for weights; the 4B needs about 18 GB
and does not fit. Gradient checkpointing uses the standard implementation,
because Unsloth's offloading version hands back bf16 activations to float32
layers ("expected mat1 and mat2 to have the same dtype").

The test split is not used here. Score the exported model locally with the
project's evaluation so all parsers are compared on the same requests.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Unsloth trains on one GPU
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from unsloth import FastLanguageModel, is_bfloat16_supported  # import before transformers
import torch, transformers, trl

print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__)
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB")
assert int(transformers.__version__.split(".")[0]) >= 5, (
    "Qwen3.5 needs transformers v5. Restart the kernel (Run > Restart) and run from this cell.")

## Settings

In [ ]:
import glob
from pathlib import Path

# NOTE for Tesla T4 (Kaggle): T4 lacks native bfloat16. For Qwen3.5's hybrid
# linear-attention, Unsloth automatically falls back to float32. In float32, the 4B
# model takes ~18 GB and OOMs on T4 (14.6 GB). Use 'Qwen/Qwen3.5-2B' (~8 GB on T4)
# OR if you need 4B on T4, set load_in_4bit = True in the next cell.
MODEL = "Qwen/Qwen3.5-2B"   # 2B fits easily on T4 in float32; use "Qwen/Qwen3.5-4B" if using 4-bit
TRIAL = True                # True: 20 steps + 10 val requests, no export
EPOCHS = 3
LR = 2e-4
MAX_LEN = 2048              # policy prompts (5 rules) reach ~1600 tokens; compiler ~1000
SEED = 3407

found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
DATA = Path(next((f for f in found if "multitask" in f), found[0])).parent
OUT = Path("/kaggle/working") / (MODEL.split("/")[-1].lower() + "-nts")
print("data:", DATA, "| output:", OUT)


## Model and LoRA adapters

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_LEN,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias = "none",
    use_gradient_checkpointing = True,   # not "unsloth": its offloading returns bf16 activations
    random_state = SEED,
    max_seq_length = MAX_LEN,
)
# Qwen3.5 is multimodal: the loader may return a processor; its text tokenizer
# is what the chat template and trainer need.
# Ensure all parameters are float32 on T4 to avoid bfloat16/float16 mismatches
model = model.to(torch.float32)

# Safety net: any linear layer that receives an input in another dtype
# (e.g. a bf16 activation recomputed by checkpointing) casts it to its own.
def _match_weight_dtype(module, args):
    if args and torch.is_tensor(args[0]) and args[0].dtype != module.weight.dtype:
        return (args[0].to(module.weight.dtype), *args[1:])

for m in model.modules():
    if isinstance(m, torch.nn.Linear):
        m.register_forward_pre_hook(_match_weight_dtype)

tok = getattr(tokenizer, "tokenizer", tokenizer)
print("eos:", repr(tok.eos_token))
print(f"GPU memory after load: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

## Data

The prompt is rendered with the chat template and **thinking disabled**, exactly
as the local server will render it, and loss is computed on the JSON answer only.

In [ ]:
import json
from datasets import load_dataset

def render_prompt(messages):
    return tok.apply_chat_template(messages[:2], tokenize=False, add_generation_prompt=True,
                                   enable_thinking=False)

def to_pc(row):
    return {"prompt": render_prompt(row["messages"]),
            "completion": row["messages"][2]["content"] + tok.eos_token}

raw = load_dataset("json", data_files={"train": str(DATA / "train.jsonl"), "val": str(DATA / "val.jsonl")})
if "task" in raw["train"].column_names:
    from collections import Counter
    print("train tasks:", Counter(raw["train"]["task"]), "| val tasks:", Counter(raw["val"]["task"]))
ds = raw.map(to_pc, remove_columns=raw["train"].column_names)
print(ds)
print(ds["train"][0]["prompt"][-400:])
print(ds["train"][0]["completion"])
n_tokens = [len(tok(p + c)["input_ids"]) for p, c in zip(ds["train"]["prompt"], ds["train"]["completion"])]
print("tokens per example: max", max(n_tokens), "mean", sum(n_tokens) // len(n_tokens))
assert max(n_tokens) <= MAX_LEN, "raise MAX_LEN"

## Train

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tok,
    train_dataset = ds["train"],
    eval_dataset = ds["val"],
    args = SFTConfig(
        max_seq_length = MAX_LEN,
        completion_only_loss = True,          # learn the JSON, not the prompt
        per_device_train_batch_size = 1,              # 1 keeps memory low on T4
        gradient_accumulation_steps = 16,             # effective batch 16
        num_train_epochs = EPOCHS,
        max_steps = 20 if TRIAL else -1,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        warmup_steps = 5,
        weight_decay = 0.01,
        optim = "adamw_8bit",
        fp16 = False,                         # Float32 on T4 (avoids BFloat16 != Half mismatch)
        bf16 = False,                         # T4 has no hardware bf16
        logging_steps = 5,
        eval_strategy = "no" if TRIAL else "epoch",
        save_strategy = "no",
        report_to = "none",
        seed = SEED,
        output_dir = str(OUT / "checkpoints"),
    ),
)
stats = trainer.train()
print(f"{stats.metrics['train_runtime'] / 60:.1f} min, "
      f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

## Quick check on the validation split

Greedy decoding on validation rows: is the output valid JSON, and is it identical
to the gold answer (per task)? Explanations are free text, so for them only the
cited fact IDs are compared. This is only a sanity check; the real scores come
from the project's evaluations (`eval_parsing --local`, `eval_policy --local`,
`eval_negotiation --local`).

In [ ]:
FastLanguageModel.for_inference(model)
from collections import defaultdict

PER_TASK = 3 if TRIAL else 25  # greedy decoding in float32 is slow; the real scores come from the local evals
val_rows, taken = [], defaultdict(int)
for line in open(DATA / "val.jsonl", encoding="utf-8"):
    row = json.loads(line)
    task = row.get("task", "compile")
    if taken[task] < PER_TASK:
        taken[task] += 1
        val_rows.append(row)

def norm(o):
    """Drop empty/default fields: [] vs missing, "none" vs missing mean the same constraint."""
    if isinstance(o, dict):
        return {k: norm(v) for k, v in o.items() if v not in (None, [], "none")}
    if isinstance(o, list):
        return [norm(x) for x in o]
    return o

def key(task, o):
    if task == "explain":  # compare what is cited, not the wording
        return sorted({i for c in o.get("claims", []) for i in c.get("facts", [])})
    if task == "policy":
        return (o.get("verdict"), sorted(o.get("cited_rules", [])), sorted(o.get("obligations", [])))
    return norm(o)

score = defaultdict(lambda: [0, 0, 0])  # n, valid JSON, match
misses = []
for row in val_rows:
    task = row.get("task", "compile")
    inputs = tok(render_prompt(row["messages"]), return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=512, do_sample=False)
    text = tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    gold = json.loads(row["messages"][2]["content"])
    score[task][0] += 1
    try:
        pred = json.loads(text)
        score[task][1] += 1
        hit = key(task, pred) == key(task, gold)
        score[task][2] += hit
        if not hit:
            misses.append((task, row["id"], text, gold))
    except json.JSONDecodeError:
        misses.append((task, row["id"], text, gold))

for task, (n, valid, match) in sorted(score.items()):
    print(f"{task:8s} n={n:4d}  valid JSON {valid / n:.0%}  match {match / n:.0%}")
for task, rid, text, gold in misses[:8]:
    print("---", task, rid)
    print("pred:", text[:300])
    print("gold:", json.dumps(gold)[:300])

## Export

* the LoRA adapter (small; keep it to re-export later);
* GGUF files for llama.cpp / Ollama: **Q5_K_M** (recommended on the 6 GB GPU,
  about 1.5 GB for 2B) and Q4_K_M (about 1.3 GB).

Unsloth merges the adapter into the 16-bit weights before quantising and builds
llama.cpp the first time, which takes several minutes.

In [ ]:
if TRIAL:
    print("TRIAL run: set TRIAL = False and run all to train fully and export.")
else:
    model.save_pretrained(str(OUT / "lora"))
    tok.save_pretrained(str(OUT / "lora"))
    model.save_pretrained_gguf(str(OUT / "gguf"), tokenizer, quantization_method=["q5_k_m", "q4_k_m"])
    for f in sorted(Path("/kaggle/working").rglob("*.gguf")):
        print(f, f"{f.stat().st_size / 2**30:.2f} GB")

## Next: run it locally

Download the `.gguf` file from the notebook's *Output* tab, then on the laptop:

```sh
llama-server -m qwen3.5-2b-nts.Q5_K_M.gguf --jinja -ngl 99 -c 8192 --port 8080
```

Thinking must stay off, as in training: send `"chat_template_kwargs": {"enable_thinking": false}`
with each request (the project's local client will do this). Keep the system prompts
and message formats unchanged (`language.compiler`, `training.distill` build them from the
same code the system runs), since the model was trained on them.